In [1]:
import os
import pandas as pd
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text, URL

In [2]:
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
project_root = Path(env_path).parent
export_dir = project_root / "exports"
export_dir.mkdir(exist_ok=True)

required_keys = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]

print("Environment file found:", bool(env_path))

for key in required_keys:
    status = "LOADED" if os.getenv(key) else "MISSING"
    print(f"{key}: {status}")



Environment file found: True
DB_HOST: LOADED
DB_PORT: LOADED
DB_NAME: LOADED
DB_USER: LOADED
DB_PASSWORD: LOADED


In [3]:
connection_url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME")
)

engine = create_engine(connection_url)

In [4]:
test_query = text("""
    SELECT COUNT(*) AS row_count
    FROM analytics.fact_order_items;
""")

with engine.connect() as connection:
    result = connection.execute(test_query)
    row_count = result.scalar_one()

print("fact_order_items row count:", row_count)

fact_order_items row count: 112650


In [5]:
sales_query = text("""
    SELECT
        f.order_id,
        f.order_item_id,
        d.full_date,
        f.quantity,
        f.price,
        f.freight_value,
        f.item_total
    FROM analytics.fact_order_items AS f
    JOIN analytics.dim_date AS d
        ON f.date_key = d.date_key
    WHERE d.full_date BETWEEN :start_date AND :end_date
    ORDER BY d.full_date, f.order_id, f.order_item_id;
""")

date_params = {
    "start_date": "2018-01-01",
    "end_date": "2018-01-31"
}

sales_df = pd.read_sql(
    sales_query,
    engine,
    params=date_params
)

sales_df.head()

,order_id,order_item_id,full_date,quantity,price,freight_value,item_total
0,00b402e8a76d3da83714c52d3977cc2d,1,2018-01-01,1,139.65,26.26,165.91
1,0462bad26ccfb8508951beb9ea398ae1,1,2018-01-01,1,151.20,18.31,169.51
2,0ab93232e2390bb66ff047c39fd91077,1,2018-01-01,1,59.99,16.86,76.85
3,0cee2d1291e4f4af44e64ffa455e15de,1,2018-01-01,1,45.00,8.27,53.27
4,1537883594710d72e0f3e96f796fcd27,1,2018-01-01,1,20.30,15.10,35.40


In [6]:
print("Row count:", len(sales_df))
print("Minimum date:", sales_df["full_date"].min())
print("Maximum date:", sales_df["full_date"].max())

sales_df.isna().sum()


Row count: 8208
Minimum date: 2018-01-01
Maximum date: 2018-01-31


order_id         0
order_item_id    0
full_date        0
quantity         0
price            0
freight_value    0
item_total       0
dtype: int64

In [7]:
totals_query = text("""
SELECT
COUNT(*) AS total_row,
SUM(quantity) AS total_quantity,
SUM(item_total) AS total_revenue
FROM analytics.fact_order_items AS f
JOIN analytics.dim_date AS d
ON f.date_key = d.date_key
WHERE d.full_date BETWEEN :start_date AND :end_date
""")

sql_totals = pd.read_sql(
    totals_query,
    engine,
    params=date_params
)
sql_totals

,total_row,total_quantity,total_revenue
0,8208,8208,1107301.89


In [8]:
pandas_total_row = len(sales_df)
pandas_total_quantity = sales_df["quantity"].sum()
pandas_total_revenue = round(sales_df["item_total"].sum(),2)

print("pandas_total_row", pandas_total_row)
print("pandas_total_quantity", pandas_total_quantity)
print("pandas_total_revenue", pandas_total_revenue)

pandas_total_row 8208
pandas_total_quantity 8208
pandas_total_revenue 1107301.89


In [9]:
sql_total_row = int(sql_totals.loc[0, "total_row"])
sql_total_quantity = int(sql_totals.loc[0, "total_quantity"])
sql_total_revenue = round(float(sql_totals.loc[0, "total_revenue"]),2)

print("sql_row_match:", sql_total_row == pandas_total_row)
print("sql_quantity_match:", sql_total_quantity == pandas_total_quantity)
print("sql_revenue_match:", sql_total_revenue == pandas_total_revenue)

sql_row_match: True
sql_quantity_match: True
sql_revenue_match: True


In [10]:
state_query = text("""
SELECT
c.customer_state,
COUNT(DISTINCT f.order_id) AS order_count,
SUM(f.quantity) AS total_quantity,
SUM(f.item_total) AS total_revenue
FROM analytics.fact_order_items AS f
    JOIN analytics.dim_customer AS c
        ON f.customer_key = c.customer_key
WHERE c.customer_state = :customer_state
GROUP BY c.customer_state
""")

In [11]:
state_params = {
    "customer_state":"SP"
}

state_df = pd.read_sql(
    state_query,
    engine,
    params = state_params
)
state_df

,customer_state,order_count,total_quantity,total_revenue
0,SP,41375,47449,5921678.12


In [12]:
def run_sales_pipeline(start_date, end_date):
    params = {
        "start_date": start_date,
        "end_date": end_date
    }

    detail_df = pd.read_sql(
        sql=sales_query,
        con=engine,
        params=params
    )

    summary_df = pd.read_sql(
        sql=totals_query,
        con=engine,
        params=params
    )

    pandas_row_count = len(detail_df)
    pandas_quantity = int(detail_df["quantity"].sum())
    pandas_total_revenue = round(
        float(detail_df["item_total"].sum()),
        2
    )

    sql_row_count = int(summary_df.loc[0, "total_row"])
    sql_total_quantity = int(summary_df.loc[0, "total_quantity"])
    sql_total_revenue = round(float(summary_df.loc[0, "total_revenue"]),
        2
    )

    assert sql_row_count == pandas_row_count, "Setir sayi uygun deyil"
    assert sql_total_quantity == pandas_quantity, "Say uygun deyil"
    assert sql_total_revenue == pandas_total_revenue, "Gelir uygun deyil"
    print("Yoxlama ugurludur")
    print("Setir sayi:", pandas_row_count)
    print("Mehsul sayi:", pandas_quantity)
    print("Umumi gelir:", pandas_total_revenue)
    csv_path = export_dir / f"sales_{start_date}_{end_date}.csv"
    excel_path = export_dir / f"sales_{start_date}_{end_date}.xlsx"

    detail_df.to_csv(csv_path, index=False)
    detail_df.to_excel(excel_path, index=False)
    print("CSV yaradildi:", csv_path)
    print("Excel yaradildi:", excel_path)

    return detail_df, summary_df

In [13]:
detail_result, summary_result = run_sales_pipeline(
    "2018-02-01",
    "2018-02-28"
)

Yoxlama ugurludur
Setir sayi: 7672
Mehsul sayi: 7672
Umumi gelir: 986908.96
CSV yaradildi: c:\Users\user\Documents\ChatGPT\Data analyst\archive\olist-product-analytics\exports\sales_2018-02-01_2018-02-28.csv
Excel yaradildi: c:\Users\user\Documents\ChatGPT\Data analyst\archive\olist-product-analytics\exports\sales_2018-02-01_2018-02-28.xlsx
